## Latent Diffusion Model

In [5]:
# ==== Robust Quant Eval (only print summary; no CSV) ==========================
import os, glob, re, math, warnings
from typing import Optional, Tuple, List
import numpy as np
from PIL import Image
import torch
import pandas as pd

# ---------------------- USER SETTINGS ----------------------
PRED_GLOB  = "./ckpt_ct2mri_b0_ldm/inference_eval/*.png"     # 예측 이미지 패턴
EVAL_ROOT  = "/data/ct-mri/dataset/processed_final_5/eval"   # eval 루트
USE_LPIPS  = True
SKIP_TRIPTYCH = True                                         # triptych 파일 스킵
# -----------------------------------------------------------

def imread_gray(path: str) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), np.float32) / 255.0

def to_uint8(x01: np.ndarray) -> np.ndarray:
    return (np.clip(x01, 0.0, 1.0) * 255.0).round().astype(np.uint8)

def psnr(a01: np.ndarray, b01: np.ndarray, data_range: float=1.0) -> float:
    mse = np.mean((a01 - b01) ** 2, dtype=np.float64)
    return float('inf') if mse <= 1e-12 else 20.0*math.log10(data_range) - 10.0*math.log10(mse)

def ssim_gray(a01: np.ndarray, b01: np.ndarray, data_range: float=1.0) -> float:
    try:
        from skimage.metrics import structural_similarity as ssim_sk
        return float(ssim_sk(a01, b01, data_range=data_range))
    except Exception:
        a=a01.astype(np.float64); b=b01.astype(np.float64)
        mu_a=a.mean(); mu_b=b.mean()
        sigma_a=a.var(); sigma_b=b.var()
        sigma_ab=((a-mu_a)*(b-mu_b)).mean()
        C1=(0.01*data_range)**2; C2=(0.03*data_range)**2
        return float(((2*mu_a*mu_b+C1)*(2*sigma_ab+C2))/((mu_a**2+mu_b**2+C1)*(sigma_a+sigma_b+C2)+1e-12))

# LPIPS 로더(가능하면)
LPIPS_FN=None
if USE_LPIPS:
    try:
        try:
            from lpips import LPIPS
            LPIPS_FN = LPIPS(net='vgg').eval()
        except Exception:
            from taming.modules.losses.lpips import LPIPS as TamingLPIPS
            LPIPS_FN = TamingLPIPS().eval()
        print("[LPIPS] loaded")
    except Exception as e:
        warnings.warn(f"[LPIPS] unavailable: {e}")
        LPIPS_FN=None

@torch.no_grad()
def lpips_val(a01: np.ndarray, b01: np.ndarray) -> Optional[float]:
    if LPIPS_FN is None: return None
    a = torch.from_numpy(a01)[None,None,:,:]*2.0-1.0
    b = torch.from_numpy(b01)[None,None,:,:]*2.0-1.0
    v = LPIPS_FN(a.repeat(1,3,1,1), b.repeat(1,3,1,1))
    return float(v.item())

def strip_prefixes(name: str) -> str:
    """ ep###_, ckpt step prefix 같은 접두어 제거 """
    name = re.sub(r"^ep\d+_", "", name)
    return name

def parse_pid_stem_from_pred(pred_path: str) -> Tuple[str,str]:
    """
    강건 파서:
      - ep###_ 접두어 제거
      - ddpm/ddim 태그가 붙어도 stem(숫자)만 정확히 추출
      - PID는 stem 바로 앞 토큰들 모두
    예:
      CNUH_001_0002_ddim_eta0.0.png            -> (CNUH_001, 0002)
      ep025_CNUH_001_0002_ddim_eta0.0.png     -> (CNUH_001, 0002)
      CNUH_X_003_0123_ddpm.png                -> (CNUH_X_003, 0123)
      CNUH_001_0002.png                       -> (CNUH_001, 0002)
    """
    base = os.path.splitext(os.path.basename(pred_path))[0]
    base = strip_prefixes(base)

    # 1) 먼저 ddpm/ddim 태그 앞부분만 사용
    base = re.split(r"_(?:ddpm|ddim)\b", base)[0]

    # 2) 끝에서 첫 번째 '숫자만' 토큰을 stem으로
    parts = base.split("_")
    stem_idx=None
    for i in range(len(parts)-1, -1, -1):
        if parts[i].isdigit():
            stem_idx=i; break
    if stem_idx is not None and stem_idx>=1:
        pid="_".join(parts[:stem_idx])
        stem=parts[stem_idx]
        return pid, stem

    # 3) 정규식 백업: ..._(\d{2,6})
    m = re.match(r"^(?P<pid>.+?)_(?P<stem>\d{2,6})$", base)
    if m:
        return m.group("pid"), m.group("stem")

    # 4) 최후 보루: stem만
    return "", parts[-1]

def try_gt_candidates(eval_root: str, pid: str, stem: str) -> Optional[str]:
    """
    eval/<PID>/{mri_b0_axial|mri_b0|mri_b0_ax}/<STEM>.(jpg|png)
    없으면 stem만 재귀 fallback
    """
    folders = ["mri_b0_axial", "mri_b0", "mri_b0_ax"]
    exts    = [".jpg", ".png", ".JPG", ".PNG"]

    if pid:
        for d in folders:
            base = os.path.join(eval_root, pid, d)
            for ext in exts:
                p = os.path.join(base, stem+ext)
                if os.path.exists(p): return p

    # stem만으로 재귀 검색
    for d in folders:
        for ext in exts:
            hits = glob.glob(os.path.join(eval_root, "**", d, stem+ext), recursive=True)
            if hits: return hits[0]
    return None

def resize_pred_to_gt(a01: np.ndarray, b01: np.ndarray) -> Tuple[np.ndarray,np.ndarray]:
    if a01.shape == b01.shape: return a01, b01
    a_img = Image.fromarray(to_uint8(a01), mode="L").resize((b01.shape[1], b01.shape[0]), Image.BICUBIC)
    return np.asarray(a_img, np.float32)/255.0, b01

# ---- main eval loop ----
all_preds = sorted(glob.glob(PRED_GLOB))
if SKIP_TRIPTYCH:
    pred_paths = [p for p in all_preds if "triptych" not in os.path.basename(p)]
else:
    pred_paths = all_preds
assert len(pred_paths) > 0, f"No predicted images matched: {PRED_GLOB}"

rows=[]; n_found=0; n_skip=0; debug_misses=[]
for p in pred_paths:
    pid, stem = parse_pid_stem_from_pred(p)
    gt = try_gt_candidates(EVAL_ROOT, pid, stem)
    if gt is None:
        n_skip += 1
        debug_misses.append((os.path.basename(p), pid or "<empty>", stem))
        continue
    try:
        a01 = imread_gray(p); b01 = imread_gray(gt)
        a01, b01 = resize_pred_to_gt(a01, b01)
        mae  = float(np.mean(np.abs(a01 - b01), dtype=np.float64))
        rmse = float(np.sqrt(np.mean((a01 - b01) ** 2, dtype=np.float64)))
        psnr_v = psnr(a01, b01, 1.0)
        ssim_v = ssim_gray(a01, b01, 1.0)
        lpips_v = lpips_val(a01, b01) if LPIPS_FN is not None else None
        rows.append({"PSNR":psnr_v,"SSIM":ssim_v,"MAE":mae,"RMSE":rmse,"LPIPS":lpips_v})
        n_found += 1
    except Exception as e:
        warnings.warn(f"[SKIP] {p} -> {e}"); n_skip += 1

if n_found == 0:
    print("---- DEBUG: no pair matched ----")
    print(f"Checked {len(pred_paths)} preds; skipped {n_skip}.")
    for j,(name,pid,stem) in enumerate(debug_misses[:10]):
        print(f"miss[{j}]: {name} => pid: {pid} stem: {stem}")
    raise RuntimeError("No pairs evaluated. Check naming/paths; see debug above.")

df = pd.DataFrame(rows)
summary = {
    "N": n_found,
    "PSNR_mean": df["PSNR"].mean(), "PSNR_std": df["PSNR"].std(),
    "SSIM_mean": df["SSIM"].mean(), "SSIM_std": df["SSIM"].std(),
    "MAE_mean":  df["MAE"].mean(),  "MAE_std":  df["MAE"].std(),
    "RMSE_mean": df["RMSE"].mean(), "RMSE_std": df["RMSE"].std(),
}
if "LPIPS" in df.columns and df["LPIPS"].notna().any():
    summary["LPIPS_mean"] = df["LPIPS"].mean(skipna=True)
    summary["LPIPS_std"]  = df["LPIPS"].std(skipna=True)

print("---- Summary ----")
for k,v in summary.items():
    print(f"{k}: {v}")


loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
[LPIPS] loaded
---- Summary ----
N: 93
PSNR_mean: 10.283047855983021
PSNR_std: 4.685449895073457
SSIM_mean: 0.2264508481360344
SSIM_std: 0.22211195680053572
MAE_mean: 0.28783601056811053
MAE_std: 0.19228930677035877
RMSE_mean: 0.3501056415125401
RMSE_std: 0.17314993370850615
LPIPS_mean: 0.499355907722186
LPIPS_std: 0.16899681209446407


## Pixel Diffusion Model

In [6]:
# ==== Robust Quant Eval (only print summary; no CSV) ==========================
import os, glob, re, math, warnings
from typing import Optional, Tuple, List
import numpy as np
from PIL import Image
import torch
import pandas as pd

# ---------------------- USER SETTINGS ----------------------
PRED_GLOB  = "./piexl_diff_ckpt_ct2mri_b0/inference_eval/*.png"     # 예측 이미지 패턴
EVAL_ROOT  = "/data/ct-mri/dataset/processed_final_5/eval"   # eval 루트
USE_LPIPS  = True
SKIP_TRIPTYCH = True                                         # triptych 파일 스킵
# -----------------------------------------------------------

def imread_gray(path: str) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), np.float32) / 255.0

def to_uint8(x01: np.ndarray) -> np.ndarray:
    return (np.clip(x01, 0.0, 1.0) * 255.0).round().astype(np.uint8)

def psnr(a01: np.ndarray, b01: np.ndarray, data_range: float=1.0) -> float:
    mse = np.mean((a01 - b01) ** 2, dtype=np.float64)
    return float('inf') if mse <= 1e-12 else 20.0*math.log10(data_range) - 10.0*math.log10(mse)

def ssim_gray(a01: np.ndarray, b01: np.ndarray, data_range: float=1.0) -> float:
    try:
        from skimage.metrics import structural_similarity as ssim_sk
        return float(ssim_sk(a01, b01, data_range=data_range))
    except Exception:
        a=a01.astype(np.float64); b=b01.astype(np.float64)
        mu_a=a.mean(); mu_b=b.mean()
        sigma_a=a.var(); sigma_b=b.var()
        sigma_ab=((a-mu_a)*(b-mu_b)).mean()
        C1=(0.01*data_range)**2; C2=(0.03*data_range)**2
        return float(((2*mu_a*mu_b+C1)*(2*sigma_ab+C2))/((mu_a**2+mu_b**2+C1)*(sigma_a+sigma_b+C2)+1e-12))

# LPIPS 로더(가능하면)
LPIPS_FN=None
if USE_LPIPS:
    try:
        try:
            from lpips import LPIPS
            LPIPS_FN = LPIPS(net='vgg').eval()
        except Exception:
            from taming.modules.losses.lpips import LPIPS as TamingLPIPS
            LPIPS_FN = TamingLPIPS().eval()
        print("[LPIPS] loaded")
    except Exception as e:
        warnings.warn(f"[LPIPS] unavailable: {e}")
        LPIPS_FN=None

@torch.no_grad()
def lpips_val(a01: np.ndarray, b01: np.ndarray) -> Optional[float]:
    if LPIPS_FN is None: return None
    a = torch.from_numpy(a01)[None,None,:,:]*2.0-1.0
    b = torch.from_numpy(b01)[None,None,:,:]*2.0-1.0
    v = LPIPS_FN(a.repeat(1,3,1,1), b.repeat(1,3,1,1))
    return float(v.item())

def strip_prefixes(name: str) -> str:
    """ ep###_, ckpt step prefix 같은 접두어 제거 """
    name = re.sub(r"^ep\d+_", "", name)
    return name

def parse_pid_stem_from_pred(pred_path: str) -> Tuple[str,str]:
    """
    강건 파서:
      - ep###_ 접두어 제거
      - ddpm/ddim 태그가 붙어도 stem(숫자)만 정확히 추출
      - PID는 stem 바로 앞 토큰들 모두
    예:
      CNUH_001_0002_ddim_eta0.0.png            -> (CNUH_001, 0002)
      ep025_CNUH_001_0002_ddim_eta0.0.png     -> (CNUH_001, 0002)
      CNUH_X_003_0123_ddpm.png                -> (CNUH_X_003, 0123)
      CNUH_001_0002.png                       -> (CNUH_001, 0002)
    """
    base = os.path.splitext(os.path.basename(pred_path))[0]
    base = strip_prefixes(base)

    # 1) 먼저 ddpm/ddim 태그 앞부분만 사용
    base = re.split(r"_(?:ddpm|ddim)\b", base)[0]

    # 2) 끝에서 첫 번째 '숫자만' 토큰을 stem으로
    parts = base.split("_")
    stem_idx=None
    for i in range(len(parts)-1, -1, -1):
        if parts[i].isdigit():
            stem_idx=i; break
    if stem_idx is not None and stem_idx>=1:
        pid="_".join(parts[:stem_idx])
        stem=parts[stem_idx]
        return pid, stem

    # 3) 정규식 백업: ..._(\d{2,6})
    m = re.match(r"^(?P<pid>.+?)_(?P<stem>\d{2,6})$", base)
    if m:
        return m.group("pid"), m.group("stem")

    # 4) 최후 보루: stem만
    return "", parts[-1]

def try_gt_candidates(eval_root: str, pid: str, stem: str) -> Optional[str]:
    """
    eval/<PID>/{mri_b0_axial|mri_b0|mri_b0_ax}/<STEM>.(jpg|png)
    없으면 stem만 재귀 fallback
    """
    folders = ["mri_b0_axial", "mri_b0", "mri_b0_ax"]
    exts    = [".jpg", ".png", ".JPG", ".PNG"]

    if pid:
        for d in folders:
            base = os.path.join(eval_root, pid, d)
            for ext in exts:
                p = os.path.join(base, stem+ext)
                if os.path.exists(p): return p

    # stem만으로 재귀 검색
    for d in folders:
        for ext in exts:
            hits = glob.glob(os.path.join(eval_root, "**", d, stem+ext), recursive=True)
            if hits: return hits[0]
    return None

def resize_pred_to_gt(a01: np.ndarray, b01: np.ndarray) -> Tuple[np.ndarray,np.ndarray]:
    if a01.shape == b01.shape: return a01, b01
    a_img = Image.fromarray(to_uint8(a01), mode="L").resize((b01.shape[1], b01.shape[0]), Image.BICUBIC)
    return np.asarray(a_img, np.float32)/255.0, b01

# ---- main eval loop ----
all_preds = sorted(glob.glob(PRED_GLOB))
if SKIP_TRIPTYCH:
    pred_paths = [p for p in all_preds if "triptych" not in os.path.basename(p)]
else:
    pred_paths = all_preds
assert len(pred_paths) > 0, f"No predicted images matched: {PRED_GLOB}"

rows=[]; n_found=0; n_skip=0; debug_misses=[]
for p in pred_paths:
    pid, stem = parse_pid_stem_from_pred(p)
    gt = try_gt_candidates(EVAL_ROOT, pid, stem)
    if gt is None:
        n_skip += 1
        debug_misses.append((os.path.basename(p), pid or "<empty>", stem))
        continue
    try:
        a01 = imread_gray(p); b01 = imread_gray(gt)
        a01, b01 = resize_pred_to_gt(a01, b01)
        mae  = float(np.mean(np.abs(a01 - b01), dtype=np.float64))
        rmse = float(np.sqrt(np.mean((a01 - b01) ** 2, dtype=np.float64)))
        psnr_v = psnr(a01, b01, 1.0)
        ssim_v = ssim_gray(a01, b01, 1.0)
        lpips_v = lpips_val(a01, b01) if LPIPS_FN is not None else None
        rows.append({"PSNR":psnr_v,"SSIM":ssim_v,"MAE":mae,"RMSE":rmse,"LPIPS":lpips_v})
        n_found += 1
    except Exception as e:
        warnings.warn(f"[SKIP] {p} -> {e}"); n_skip += 1

if n_found == 0:
    print("---- DEBUG: no pair matched ----")
    print(f"Checked {len(pred_paths)} preds; skipped {n_skip}.")
    for j,(name,pid,stem) in enumerate(debug_misses[:10]):
        print(f"miss[{j}]: {name} => pid: {pid} stem: {stem}")
    raise RuntimeError("No pairs evaluated. Check naming/paths; see debug above.")

df = pd.DataFrame(rows)
summary = {
    "N": n_found,
    "PSNR_mean": df["PSNR"].mean(), "PSNR_std": df["PSNR"].std(),
    "SSIM_mean": df["SSIM"].mean(), "SSIM_std": df["SSIM"].std(),
    "MAE_mean":  df["MAE"].mean(),  "MAE_std":  df["MAE"].std(),
    "RMSE_mean": df["RMSE"].mean(), "RMSE_std": df["RMSE"].std(),
}
if "LPIPS" in df.columns and df["LPIPS"].notna().any():
    summary["LPIPS_mean"] = df["LPIPS"].mean(skipna=True)
    summary["LPIPS_std"]  = df["LPIPS"].std(skipna=True)

print("---- Summary ----")
for k,v in summary.items():
    print(f"{k}: {v}")


/home/youkyoung/.conda/envs/med/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/youkyoung/.conda/envs/med/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/youkyoung/CT2MRI/diffusion_model/taming/modules/losses/lpips.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github

loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
[LPIPS] loaded
---- Summary ----
N: 93
PSNR_mean: 7.941447447345868
PSNR_std: 2.2762369155021616
SSIM_mean: 0.053488470354410234
SSIM_std: 0.0365217456363083
MAE_mean: 0.37904914848210913
MAE_std: 0.10561745713417817
RMSE_mean: 0.4139110398234745
RMSE_std: 0.10166468083842134
LPIPS_mean: 0.671496573955782
LPIPS_std: 0.04977499528060208


## Feature Alignment

In [7]:
# ==== Robust Quant Eval (only print summary; no CSV) ==========================
import os, glob, re, math, warnings
from typing import Optional, Tuple, List
import numpy as np
from PIL import Image
import torch
import pandas as pd

# ---------------------- USER SETTINGS ----------------------
PRED_GLOB  = "./ckpt_ct2mri_b0_ldm_align_bucket_preinit/inference_eval/*.png"     # 예측 이미지 패턴
EVAL_ROOT  = "/data/ct-mri/dataset/processed_final_5/eval"   # eval 루트
USE_LPIPS  = True
SKIP_TRIPTYCH = True                                         # triptych 파일 스킵
# -----------------------------------------------------------

def imread_gray(path: str) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), np.float32) / 255.0

def to_uint8(x01: np.ndarray) -> np.ndarray:
    return (np.clip(x01, 0.0, 1.0) * 255.0).round().astype(np.uint8)

def psnr(a01: np.ndarray, b01: np.ndarray, data_range: float=1.0) -> float:
    mse = np.mean((a01 - b01) ** 2, dtype=np.float64)
    return float('inf') if mse <= 1e-12 else 20.0*math.log10(data_range) - 10.0*math.log10(mse)

def ssim_gray(a01: np.ndarray, b01: np.ndarray, data_range: float=1.0) -> float:
    try:
        from skimage.metrics import structural_similarity as ssim_sk
        return float(ssim_sk(a01, b01, data_range=data_range))
    except Exception:
        a=a01.astype(np.float64); b=b01.astype(np.float64)
        mu_a=a.mean(); mu_b=b.mean()
        sigma_a=a.var(); sigma_b=b.var()
        sigma_ab=((a-mu_a)*(b-mu_b)).mean()
        C1=(0.01*data_range)**2; C2=(0.03*data_range)**2
        return float(((2*mu_a*mu_b+C1)*(2*sigma_ab+C2))/((mu_a**2+mu_b**2+C1)*(sigma_a+sigma_b+C2)+1e-12))

# LPIPS 로더(가능하면)
LPIPS_FN=None
if USE_LPIPS:
    try:
        try:
            from lpips import LPIPS
            LPIPS_FN = LPIPS(net='vgg').eval()
        except Exception:
            from taming.modules.losses.lpips import LPIPS as TamingLPIPS
            LPIPS_FN = TamingLPIPS().eval()
        print("[LPIPS] loaded")
    except Exception as e:
        warnings.warn(f"[LPIPS] unavailable: {e}")
        LPIPS_FN=None

@torch.no_grad()
def lpips_val(a01: np.ndarray, b01: np.ndarray) -> Optional[float]:
    if LPIPS_FN is None: return None
    a = torch.from_numpy(a01)[None,None,:,:]*2.0-1.0
    b = torch.from_numpy(b01)[None,None,:,:]*2.0-1.0
    v = LPIPS_FN(a.repeat(1,3,1,1), b.repeat(1,3,1,1))
    return float(v.item())

def strip_prefixes(name: str) -> str:
    """ ep###_, ckpt step prefix 같은 접두어 제거 """
    name = re.sub(r"^ep\d+_", "", name)
    return name

def parse_pid_stem_from_pred(pred_path: str) -> Tuple[str,str]:
    """
    강건 파서:
      - ep###_ 접두어 제거
      - ddpm/ddim 태그가 붙어도 stem(숫자)만 정확히 추출
      - PID는 stem 바로 앞 토큰들 모두
    예:
      CNUH_001_0002_ddim_eta0.0.png            -> (CNUH_001, 0002)
      ep025_CNUH_001_0002_ddim_eta0.0.png     -> (CNUH_001, 0002)
      CNUH_X_003_0123_ddpm.png                -> (CNUH_X_003, 0123)
      CNUH_001_0002.png                       -> (CNUH_001, 0002)
    """
    base = os.path.splitext(os.path.basename(pred_path))[0]
    base = strip_prefixes(base)

    # 1) 먼저 ddpm/ddim 태그 앞부분만 사용
    base = re.split(r"_(?:ddpm|ddim)\b", base)[0]

    # 2) 끝에서 첫 번째 '숫자만' 토큰을 stem으로
    parts = base.split("_")
    stem_idx=None
    for i in range(len(parts)-1, -1, -1):
        if parts[i].isdigit():
            stem_idx=i; break
    if stem_idx is not None and stem_idx>=1:
        pid="_".join(parts[:stem_idx])
        stem=parts[stem_idx]
        return pid, stem

    # 3) 정규식 백업: ..._(\d{2,6})
    m = re.match(r"^(?P<pid>.+?)_(?P<stem>\d{2,6})$", base)
    if m:
        return m.group("pid"), m.group("stem")

    # 4) 최후 보루: stem만
    return "", parts[-1]

def try_gt_candidates(eval_root: str, pid: str, stem: str) -> Optional[str]:
    """
    eval/<PID>/{mri_b0_axial|mri_b0|mri_b0_ax}/<STEM>.(jpg|png)
    없으면 stem만 재귀 fallback
    """
    folders = ["mri_b0_axial", "mri_b0", "mri_b0_ax"]
    exts    = [".jpg", ".png", ".JPG", ".PNG"]

    if pid:
        for d in folders:
            base = os.path.join(eval_root, pid, d)
            for ext in exts:
                p = os.path.join(base, stem+ext)
                if os.path.exists(p): return p

    # stem만으로 재귀 검색
    for d in folders:
        for ext in exts:
            hits = glob.glob(os.path.join(eval_root, "**", d, stem+ext), recursive=True)
            if hits: return hits[0]
    return None

def resize_pred_to_gt(a01: np.ndarray, b01: np.ndarray) -> Tuple[np.ndarray,np.ndarray]:
    if a01.shape == b01.shape: return a01, b01
    a_img = Image.fromarray(to_uint8(a01), mode="L").resize((b01.shape[1], b01.shape[0]), Image.BICUBIC)
    return np.asarray(a_img, np.float32)/255.0, b01

# ---- main eval loop ----
all_preds = sorted(glob.glob(PRED_GLOB))
if SKIP_TRIPTYCH:
    pred_paths = [p for p in all_preds if "triptych" not in os.path.basename(p)]
else:
    pred_paths = all_preds
assert len(pred_paths) > 0, f"No predicted images matched: {PRED_GLOB}"

rows=[]; n_found=0; n_skip=0; debug_misses=[]
for p in pred_paths:
    pid, stem = parse_pid_stem_from_pred(p)
    gt = try_gt_candidates(EVAL_ROOT, pid, stem)
    if gt is None:
        n_skip += 1
        debug_misses.append((os.path.basename(p), pid or "<empty>", stem))
        continue
    try:
        a01 = imread_gray(p); b01 = imread_gray(gt)
        a01, b01 = resize_pred_to_gt(a01, b01)
        mae  = float(np.mean(np.abs(a01 - b01), dtype=np.float64))
        rmse = float(np.sqrt(np.mean((a01 - b01) ** 2, dtype=np.float64)))
        psnr_v = psnr(a01, b01, 1.0)
        ssim_v = ssim_gray(a01, b01, 1.0)
        lpips_v = lpips_val(a01, b01) if LPIPS_FN is not None else None
        rows.append({"PSNR":psnr_v,"SSIM":ssim_v,"MAE":mae,"RMSE":rmse,"LPIPS":lpips_v})
        n_found += 1
    except Exception as e:
        warnings.warn(f"[SKIP] {p} -> {e}"); n_skip += 1

if n_found == 0:
    print("---- DEBUG: no pair matched ----")
    print(f"Checked {len(pred_paths)} preds; skipped {n_skip}.")
    for j,(name,pid,stem) in enumerate(debug_misses[:10]):
        print(f"miss[{j}]: {name} => pid: {pid} stem: {stem}")
    raise RuntimeError("No pairs evaluated. Check naming/paths; see debug above.")

df = pd.DataFrame(rows)
summary = {
    "N": n_found,
    "PSNR_mean": df["PSNR"].mean(), "PSNR_std": df["PSNR"].std(),
    "SSIM_mean": df["SSIM"].mean(), "SSIM_std": df["SSIM"].std(),
    "MAE_mean":  df["MAE"].mean(),  "MAE_std":  df["MAE"].std(),
    "RMSE_mean": df["RMSE"].mean(), "RMSE_std": df["RMSE"].std(),
}
if "LPIPS" in df.columns and df["LPIPS"].notna().any():
    summary["LPIPS_mean"] = df["LPIPS"].mean(skipna=True)
    summary["LPIPS_std"]  = df["LPIPS"].std(skipna=True)

print("---- Summary ----")
for k,v in summary.items():
    print(f"{k}: {v}")


/home/youkyoung/.conda/envs/med/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/youkyoung/.conda/envs/med/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/youkyoung/CT2MRI/diffusion_model/taming/modules/losses/lpips.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github

loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
[LPIPS] loaded
---- Summary ----
N: 93
PSNR_mean: 14.624013684014667
PSNR_std: 1.5481705943035966
SSIM_mean: 0.5639643370114
SSIM_std: 0.06992009333046921
MAE_mean: 0.10672770457878776
MAE_std: 0.025904099045444735
RMSE_mean: 0.18863295624203996
RMSE_std: 0.03373327443018709
LPIPS_mean: 0.4177978365652023
LPIPS_std: 0.04327818095738656
